In [37]:
import pandas as pd
import numpy as np

# Training Data

In [ ]:
features = [
    "speed",
    "temperature",
    "density",
    "bz_gsm",
    "bx_gsm",
    "by_gsm",
    "pressure",
    "smoothed_ssn",
]

# load
solar_train = pd.read_csv("../../data/model/train/raw/solar_wind.csv")  # solar wind
sun_train = pd.read_csv("../../data/model/train/raw/sunspots.csv")  # sunspots
dst_train = pd.read_csv("../../data/model/train/raw/dst_labels.csv")  # dst

In [39]:

# convert timedelta
solar_train["timedelta"] = pd.to_timedelta(solar_train["timedelta"])
sun_train["timedelta"] = pd.to_timedelta(sun_train["timedelta"])

sun_train.sort_values(["period", "timedelta"], inplace=True)

# pressure column
solar_train["pressure"] = solar_train["density"] * solar_train["speed"] ** 2

# merge
solar_train["days"] = solar_train["timedelta"].dt.days
sun_train["days"] = sun_train["timedelta"].dt.days
solar_train = pd.merge(solar_train, sun_train[["period", "days", "smoothed_ssn"]], "left", ["period", "days"])
solar_train.drop(columns="days", inplace=True)

solar_train.sort_values(["period", "timedelta"], inplace=True)
solar_train.reset_index(inplace=True)

# drop unimportant columns
solar_train = solar_train[["period", "timedelta"] + features]

# fill blanks
solar_train[['speed', 'density', 'temperature']] = solar_train[['speed', 'density', 'temperature']].replace(0, np.nan)

train_short = [c for c in features if c != "smoothed_ssn"]
for p in ["train_a", "train_b", "train_c"]:
    curr_period = solar_train["period"] == p

    solar_train.loc[curr_period, "smoothed_ssn"] = (
        solar_train.loc[curr_period, "smoothed_ssn"].ffill().bfill()
    )
    roll = (
        solar_train[train_short]
        .rolling(window=20, min_periods=5)
        .mean()
        .interpolate("linear", axis=0)
    )
    solar_train.loc[curr_period, train_short] = solar_train.loc[curr_period, train_short].fillna(roll)
    solar_train.loc[curr_period, train_short] = solar_train.loc[curr_period, train_short].ffill().bfill()

# normalize
median = solar_train[features].median()
uq = solar_train[features].quantile(0.75)
lq = solar_train[features].quantile(0.25)
iqr = uq - lq
solar_train[features] = (solar_train[features] - median) / iqr

stats = pd.DataFrame({"Median": median, "Iqr": iqr})

median = dst_train["dst"].median()
uq = dst_train["dst"].quantile(0.75)
lq = dst_train["dst"].quantile(0.25)
iqr = uq - lq

stats.loc[len(stats)] = {"Median": median, "Iqr": iqr}

In [40]:
stats.to_csv("../../data/model/train/transformed/stats.csv", index=False)

In [50]:
a, b = stats.iloc[-1]  # dst label median and iqr
# a
solar_train_a = solar_train[solar_train["period"] == "train_a"][["timedelta"] + features]
solar_train_a.reset_index(inplace=True, drop=True)
solar_train_a["timedelta"] = pd.to_timedelta(solar_train_a["timedelta"])
solar_train_a = solar_train_a.set_index("timedelta").resample("h").agg(["mean", "std"])
solar_train_a.columns = ["_".join(col) for col in solar_train_a]
solar_train_a.drop("smoothed_ssn_std", axis=1, inplace=True)

dst_train_a = dst_train[dst_train["period"] == "train_a"]["dst"]
dst_train_a = (dst_train_a - a) / b

solar_train_a['dst'] = dst_train_a.values

# b
solar_train_b = solar_train[solar_train["period"] == "train_b"][["timedelta"] + features]
solar_train_b.reset_index(inplace=True, drop=True)
solar_train_b["timedelta"] = pd.to_timedelta(solar_train_b["timedelta"])
solar_train_b = solar_train_b.set_index("timedelta").resample("h").agg(["mean", "std"])
solar_train_b.columns = ["_".join(col) for col in solar_train_b]
solar_train_b.drop("smoothed_ssn_std", axis=1, inplace=True)

dst_train_b = dst_train[dst_train["period"] == "train_b"]["dst"]
dst_train_b = (dst_train_b - a) / b

solar_train_b['dst'] = dst_train_b.values

# c
solar_train_c = solar_train[solar_train["period"] == "train_c"][["timedelta"] + features]
solar_train_c.reset_index(inplace=True, drop=True)
solar_train_c["timedelta"] = pd.to_timedelta(solar_train_c["timedelta"])
solar_train_c = solar_train_c.set_index("timedelta").resample("h").agg(["mean", "std"])
solar_train_c.columns = ["_".join(col) for col in solar_train_c]
solar_train_c.drop("smoothed_ssn_std", axis=1, inplace=True)

dst_train_c = dst_train[dst_train["period"] == "train_c"]["dst"]
dst_train_c = (dst_train_c - a) / b

solar_train_c['dst'] = dst_train_c.values

In [ ]:
solar_train_a.to_csv('../../data/model/train/transformed/solar_train_a')
solar_train_b.to_csv('../../data/model/train/transformed/solar_train_b')
solar_train_c.to_csv('../../data/model/train/transformed/solar_train_c')

# Validation Data

In [65]:
# load
solar_val = pd.read_csv("../../data/model/validation/raw/solar_wind.csv")  # solar wind
sun_val = pd.read_csv("../../data/model/validation/raw/sunspots.csv")  # sunspots
dst_val = pd.read_csv("../../data/model/validation/raw/dst_labels.csv")  # dst

In [66]:
# convert timedelta
solar_val["timedelta"] = pd.to_timedelta(solar_val["timedelta"])
sun_val["timedelta"] = pd.to_timedelta(sun_val["timedelta"])

sun_val.sort_values(["period", "timedelta"], inplace=True)

# pressure column
solar_val["pressure"] = solar_val["density"] * solar_val["speed"] ** 2

# merge
solar_val["days"] = solar_val["timedelta"].dt.days
sun_val["days"] = sun_val["timedelta"].dt.days
solar_val = pd.merge(solar_val, sun_val[["period", "days", "smoothed_ssn"]], "left", ["period", "days"])
solar_val.drop(columns="days", inplace=True)

solar_val.sort_values(["period", "timedelta"], inplace=True)
solar_val.reset_index(inplace=True)

# drop unimportant columns
solar_val = solar_val[["period", "timedelta"] + features]

# fill blanks
solar_val[['speed', 'density', 'temperature']] = solar_val[['speed', 'density', 'temperature']].replace(0, np.nan)

val_short = [c for c in features if c != "smoothed_ssn"]
for p in ["test_a", "test_b", "test_c"]:
    curr_period = solar_val["period"] == p

    solar_val.loc[curr_period, "smoothed_ssn"] = (
        solar_val.loc[curr_period, "smoothed_ssn"].ffill().bfill()
    )
    roll = (
        solar_val[val_short]
        .rolling(window=20, min_periods=5)
        .mean()
        .interpolate("linear", axis=0)
    )
    solar_val.loc[curr_period, val_short] = solar_val.loc[curr_period, val_short].fillna(roll)
    solar_val.loc[curr_period, val_short] = solar_val.loc[curr_period, val_short].ffill().bfill()

In [69]:
a, b = stats.iloc[-1]  # dst label median and iqr
M = stats["Median"][:-1].values
I = stats["Iqr"][:-1].values

solar_val[features] = (solar_val[features] - M) / I
# a
solar_val_a = solar_val[solar_val["period"] == "test_a"][["timedelta"] + features]
solar_val_a.reset_index(inplace=True, drop=True)
solar_val_a["timedelta"] = pd.to_timedelta(solar_val_a["timedelta"])
solar_val_a = solar_val_a.set_index("timedelta").resample("h").agg(["mean", "std"])
solar_val_a.columns = ["_".join(col) for col in solar_val_a]
solar_val_a.drop("smoothed_ssn_std", axis=1, inplace=True)

dst_val_a = dst_val[dst_val["period"] == "test_a"]["dst"]
dst_val_a = (dst_val_a - a) / b

solar_val_a['dst'] = dst_val_a.values

# b
solar_val_b = solar_val[solar_val["period"] == "test_b"][["timedelta"] + features]
solar_val_b.reset_index(inplace=True, drop=True)
solar_val_b["timedelta"] = pd.to_timedelta(solar_val_b["timedelta"])
solar_val_b = solar_val_b.set_index("timedelta").resample("h").agg(["mean", "std"])
solar_val_b.columns = ["_".join(col) for col in solar_val_b]
solar_val_b.drop("smoothed_ssn_std", axis=1, inplace=True)

dst_val_b = dst_val[dst_val["period"] == "test_b"]["dst"]
dst_val_b = (dst_val_b - a) / b

solar_val_b['dst'] = dst_val_b.values

# c
solar_val_c = solar_val[solar_val["period"] == "test_c"][["timedelta"] + features]
solar_val_c.reset_index(inplace=True, drop=True)
solar_val_c["timedelta"] = pd.to_timedelta(solar_val_c["timedelta"])
solar_val_c = solar_val_c.set_index("timedelta").resample("h").agg(["mean", "std"])
solar_val_c.columns = ["_".join(col) for col in solar_val_c]
solar_val_c.drop("smoothed_ssn_std", axis=1, inplace=True)

dst_val_c = dst_val[dst_val["period"] == "test_c"]["dst"]
dst_val_c = (dst_val_c - a) / b

solar_val_c['dst'] = dst_val_c.values

In [70]:
solar_val_a.to_csv('../../data/model/validation/transformed/solar_val_a')
solar_val_b.to_csv('../../data/model/validation/transformed/solar_val_b')
solar_val_c.to_csv('../../data/model/validation/transformed/solar_val_c')